In [1]:
import math
import inspect
from collections import deque

import numpy as np

np.set_printoptions(precision=4, suppress=True)

# Task 1

## Task 1.1

Datos medidos: $N = 1\,200$, $M = 3\,850$, $\langle C \rangle = 0.48$, $\langle d \rangle = 4.2$.

### a. Grado promedio $\langle k \rangle$

In [2]:
N, M = 1200, 3850
C_medido, d_medido = 0.48, 4.2

k_prom = 2 * M / N
print(f"<k> = 2M/N = 2({M})/{N} = {2*M}/{N} = {k_prom:.4f}")

<k> = 2M/N = 2(3850)/1200 = 7700/1200 = 6.4167


### b. Valores esperados en una red Erdős-Rényi equivalente

In [3]:
p_ER = k_prom / (N - 1)
C_aleatoria = k_prom / N
d_aleatoria = math.log(N) / math.log(k_prom)

print(f"p = <k>/(N-1) = {p_ER:.6f}")
print(f"C_aleatoria = <k>/N     = {C_aleatoria:.6f}")
print(f"ln N = {math.log(N):.4f}")
print(f"ln <k> = {math.log(k_prom):.4f}")
print(f"<d>_aleatoria = ln N / ln <k> = {d_aleatoria:.4f}")

p = <k>/(N-1) = 0.005352
C_aleatoria = <k>/N     = 0.005347
ln N = 7.0901
ln <k> = 1.8589
<d>_aleatoria = ln N / ln <k> = 3.8141


### c. Comparación y clasificación topológica

In [4]:
razon_C = C_medido / C_aleatoria
razon_d = d_medido / d_aleatoria
sigma = razon_C / razon_d   # coeficiente small-world (Humphries & Gurney, 2008)

print(f"{'Métrica':<10}{'Medida':>10}{'ER equivalente':>16}{'Razón medida/ER':>18}")
print(f"{'<C>':<10}{C_medido:>10.4f}{C_aleatoria:>16.4f}{razon_C:>18.2f}")
print(f"{'<d>':<10}{d_medido:>10.4f}{d_aleatoria:>16.4f}{razon_d:>18.2f}")
print(f"\nsigma = (C/C_ER) / (d/d_ER) = {sigma:.1f}")

Métrica       Medida  ER equivalente   Razón medida/ER
<C>           0.4800          0.0053             89.77
<d>           4.2000          3.8141              1.10

sigma = (C/C_ER) / (d/d_ER) = 81.5


## Task 1.2

In [5]:
A = np.array([
    [0, 1, 1, 0, 0],
    [1, 0, 1, 1, 0],
    [1, 1, 0, 0, 1],
    [0, 1, 0, 0, 1],
    [0, 0, 1, 1, 0],
])
aristas = [(i + 1, j + 1) for i in range(len(A)) for j in range(i + 1, len(A)) if A[i, j]]
print("Simétrica:", np.array_equal(A, A.T), "| Diagonal nula:", not A.diagonal().any())
print("Aristas:", aristas, "| M =", len(aristas))

Simétrica: True | Diagonal nula: True
Aristas: [(1, 2), (1, 3), (2, 3), (2, 4), (3, 5), (4, 5)] | M = 6


## Task 1.3

In [6]:
def _validar_adyacencia(A):
    # Convierte a arreglo NumPy y verifica que sea una matriz de adyacencia no dirigida válida.
    A = np.asarray(A)
    if A.ndim != 2 or A.shape[0] != A.shape[1]:
        raise ValueError("A debe ser una matriz cuadrada")
    if not np.array_equal(A, A.T):
        raise ValueError("A debe ser simétrica (red no dirigida)")
    if A.diagonal().any():
        raise ValueError("A no debe tener lazos (diagonal nula)")
    return A


def grado(A):
    # Retorna el grado k_i de cada nodo: la suma de la fila i de A.
    A = _validar_adyacencia(A)
    return A.sum(axis=1).astype(int)


def clustering(A):
    # Retorna el coeficiente de clustering C_i = e_i / (k_i (k_i - 1) / 2) de cada nodo.
    A = _validar_adyacencia(A)
    n = A.shape[0]
    C = np.zeros(n)
    for i in range(n):
        vecinos = np.flatnonzero(A[i])
        k_i = len(vecinos)
        if k_i < 2:
            continue  # C_i no está definido; por convención se deja en 0
        e_i = A[np.ix_(vecinos, vecinos)].sum() / 2  # aristas entre los vecinos de i
        C[i] = e_i / (k_i * (k_i - 1) / 2)
    return C


def bfs(A, origen):
    # Distancias geodésicas desde `origen` a todos los nodos; -1 si no hay camino.
    A = np.asarray(A)
    dist = np.full(A.shape[0], -1, dtype=int)
    dist[origen] = 0
    cola = deque([origen])
    while cola:
        u = cola.popleft()
        for v in np.flatnonzero(A[u]):
            if dist[v] == -1:
                dist[v] = dist[u] + 1
                cola.append(v)
    return dist


def matriz_distancias(A):
    # Matriz D con D[i, j] = d_ij, obtenida con un BFS desde cada nodo.
    A = _validar_adyacencia(A)
    return np.array([bfs(A, i) for i in range(A.shape[0])])


def distancia_promedio(A):
    # Promedio de d_ij sobre los pares i < j que tienen camino entre sí.
    D = matriz_distancias(A)
    d_pares = D[np.triu_indices(D.shape[0], k=1)]
    d_pares = d_pares[d_pares > 0]  # descarta pares sin camino (-1)
    if d_pares.size == 0:
        return float("nan")
    return float(d_pares.mean())

### Verificación con la matriz del Task 1.2

In [7]:
k = grado(A)
C = clustering(A)
D = matriz_distancias(A)
d_prom = distancia_promedio(A)

print(f"{'Nodo':<6}{'k_i':>5}{'C_i':>10}")
for i in range(len(A)):
    print(f"{i+1:<6}{k[i]:>5}{C[i]:>10.4f}")
print(f"\n<k> = {k.mean():.4f}")
print(f"<C> = {C.mean():.4f}")

print("\nDistancias d_ij (triángulo superior):")
n = len(A)
print("     " + "".join(f"{j+1:>4}" for j in range(1, n)))
for i in range(n - 1):
    fila = "".join("    " if j <= i else f"{D[i, j]:>4}" for j in range(1, n))
    print(f"{i+1:>4} {fila}")
print(f"\n<d> = {d_prom:.4f}")

Nodo    k_i       C_i
1         2    1.0000
2         3    0.3333
3         3    0.3333
4         2    0.0000
5         2    0.0000

<k> = 2.4000
<C> = 0.3333

Distancias d_ij (triángulo superior):
        2   3   4   5
   1    1   1   2   2
   2        1   1   2
   3            2   1
   4                1

<d> = 1.4000


In [8]:
# Comparación automática con los resultados manuales del Task 1.2
k_manual = np.array([2, 3, 3, 2, 2])
C_manual = np.array([1, 1/3, 1/3, 0, 0])
d_manual = 14 / 10

assert np.array_equal(k, k_manual)
assert np.allclose(C, C_manual)
assert math.isclose(d_prom, d_manual)

# Verificación cruzada del clustering: (A^3)_ii = 2 * (número de triángulos que contienen a i)
triangulos = np.diag(np.linalg.matrix_power(A, 3)) / 2
C_alt = np.divide(triangulos, k * (k - 1) / 2, out=np.zeros(len(A)), where=k >= 2)
assert np.allclose(C, C_alt)

print("✔ grado, clustering y distancia_promedio coinciden con los cálculos manuales")
print("✔ clustering coincide con el cálculo alternativo por traza de A^3")

✔ grado, clustering y distancia_promedio coinciden con los cálculos manuales
✔ clustering coincide con el cálculo alternativo por traza de A^3


### Verificación de la predicción del Task 1.2d (nodo 6 conectado a 2 y 4)

In [9]:
A6 = np.zeros((6, 6), dtype=int)
A6[:5, :5] = A
for v in (2, 4):                  # nodos 2 y 4 (índices 1 y 3)
    A6[5, v - 1] = A6[v - 1, 5] = 1

k6, C6 = grado(A6), clustering(A6)
print(f"{'Nodo':<6}{'k_i antes':>10}{'k_i después':>13}{'C_i antes':>11}{'C_i después':>13}")
for i in range(6):
    ka = f"{k[i]}" if i < 5 else "—"
    Ca = f"{C[i]:.4f}" if i < 5 else "—"
    print(f"{i+1:<6}{ka:>10}{k6[i]:>13}{Ca:>11}{C6[i]:>13.4f}")
print(f"\n<k>: {k.mean():.4f} -> {k6.mean():.4f}")
print(f"<C>: {C.mean():.4f} -> {C6.mean():.4f}")
print(f"<d>: {d_prom:.4f} -> {distancia_promedio(A6):.4f}")

Nodo   k_i antes  k_i después  C_i antes  C_i después
1              2            2     1.0000       1.0000
2              3            4     0.3333       0.3333
3              3            3     0.3333       0.3333
4              2            3     0.0000       0.3333
5              2            2     0.0000       0.0000
6              —            2          —       1.0000

<k>: 2.4000 -> 2.6667
<C>: 0.3333 -> 0.5000
<d>: 1.4000 -> 1.4667


Se confirma el argumento del inciso d: $\langle k\rangle$ pasa de 2.4 a 2.667 y $\langle C\rangle$ **sube** de 0.333 a 0.5. El aumento viene del nodo 6 ($C_6 = 1$) y del nodo 4 (de 0 a 1/3). El nodo 2 se mantiene en 1/3 porque tanto el numerador como el denominador crecieron.


# Task 2

En todo el Task 2 se usa la condición de epidemia en redes heterogéneas vista en clase:

$$
\frac{\beta}{\gamma} > \lambda_c \equiv \frac{\langle k \rangle}{\langle k^2 \rangle - \langle k \rangle}
$$

$\lambda_c$ es el **umbral epidémico**. Hay brote si $\beta/\gamma$ lo supera. Para medir qué tan lejos está el sistema del umbral también se reporta la razón $(\beta/\gamma)/\lambda_c$. Si es mayor que 1, el brote se propaga. Esta razón equivale a un $R_0$ efectivo de la red:

$$
R_0^{\text{red}} = \frac{\beta}{\gamma}\cdot\frac{\langle k^2\rangle - \langle k\rangle}{\langle k\rangle}
$$

In [10]:
def umbral_epidemico(k1, k2):
    # lambda_c = <k> / (<k^2> - <k>); si <k²> <= <k> no hay componente gigante y el umbral es infinito
    if k2 - k1 <= 0:
        return float("inf")
    return k1 / (k2 - k1)


def reporte(nombre, beta, gamma, k1, k2):
    lam = umbral_epidemico(k1, k2)
    r = beta / gamma
    estado = "SE PROPAGA" if r > lam else "CONTENIDO"
    print(f"{nombre:<34} <k>={k1:8.4f}  <k²>={k2:9.4f}  λc={lam:.5f}  "
          f"β/γ={r:.4f}  (β/γ)/λc={r/lam:7.2f}  → {estado}")
    return lam

## Task 2.1

Brote con $\beta = 0.08$, $\gamma = 0.05$, $\langle k\rangle = 8.4$, $\langle k^2\rangle = 210.6$


### a. Umbral epidémico

In [11]:
beta, gamma_rec = 0.08, 0.05
k1, k2 = 8.4, 210.6

lam_base = reporte("Situación base", beta, gamma_rec, k1, k2)

Situación base                     <k>=  8.4000  <k²>= 210.6000  λc=0.04154  β/γ=1.6000  (β/γ)/λc=  38.51  → SE PROPAGA


### b. Evaluación de las estrategias A y B

In [12]:
# Estrategia A: beta se reduce 40 %; la red no cambia
lam_A = reporte("Estrategia A (β -40 %)", 0.6 * beta, gamma_rec, k1, k2)

# Estrategia B: vacunación aleatoria 30 % -> <k> -30 %, <k²> -51 %
lam_B = reporte("Estrategia B (vacunación 30 %)", beta, gamma_rec, 0.70 * k1, 0.49 * k2)

# Ambas estrategias combinadas
lam_AB = reporte("A + B combinadas", 0.6 * beta, gamma_rec, 0.70 * k1, 0.49 * k2)

Estrategia A (β -40 %)             <k>=  8.4000  <k²>= 210.6000  λc=0.04154  β/γ=0.9600  (β/γ)/λc=  23.11  → SE PROPAGA
Estrategia B (vacunación 30 %)     <k>=  5.8800  <k²>= 103.1940  λc=0.06042  β/γ=1.6000  (β/γ)/λc=  26.48  → SE PROPAGA
A + B combinadas                   <k>=  5.8800  <k²>= 103.1940  λc=0.06042  β/γ=0.9600  (β/γ)/λc=  15.89  → SE PROPAGA


In [13]:
beta_necesario = lam_base * gamma_rec
g_c_aleatoria = 1 - k1 * (1 + gamma_rec / beta) / k2

print(f"β necesario (solo A)      = {beta_necesario:.6f}  → reducción de β = {1 - beta_necesario/beta:.2%}")
print(f"g crítico (solo B, azar)  = {g_c_aleatoria:.4f}  → vacunar al azar > {g_c_aleatoria:.1%}")

β necesario (solo A)      = 0.002077  → reducción de β = 97.40%
g crítico (solo B, azar)  = 0.9352  → vacunar al azar > 93.5%


### c. Por qué la reducción del 51 % en $\langle k^2\rangle$ es consistente con la teoría, y una alternativa más eficiente

In [14]:
g = 0.3
k2_exacto = (1 - g)**2 * k2 + g * (1 - g) * k1
kappa_antes, kappa_despues = k2 / k1, (0.49 * k2) / (0.70 * k1)

print(f"<k²> tras vacunación (aprox. del enunciado): {0.49*k2:.3f}  -> λc = {umbral_epidemico(0.7*k1, 0.49*k2):.5f}")
print(f"<k²> tras vacunación (dilución binomial):     {k2_exacto:.3f}  -> λc = {umbral_epidemico(0.7*k1, k2_exacto):.5f}")
print(f"κ = <k²>/<k>: {kappa_antes:.2f} -> {kappa_despues:.2f}")
print(f"σ_k = {math.sqrt(k2 - k1**2):.2f}   CV = {math.sqrt(k2 - k1**2)/k1:.3f}   (Poisson: CV = {1/math.sqrt(k1):.3f})")
print(f"Grado esperado de un vecino (vacunación por conocidos): <k²>/<k> = {k2/k1:.2f}")

<k²> tras vacunación (aprox. del enunciado): 103.194  -> λc = 0.06042
<k²> tras vacunación (dilución binomial):     104.958  -> λc = 0.05935
κ = <k²>/<k>: 25.07 -> 17.55
σ_k = 11.83   CV = 1.409   (Poisson: CV = 0.345)
Grado esperado de un vecino (vacunación por conocidos): <k²>/<k> = 25.07


## Task 2.2 — Municipios X (Poisson) e Y (ley de potencias)

Se mantienen $\beta = 0.08$ y $\gamma = 0.05$, así que $\beta/\gamma = 1.6$.


### a. Municipio X: distribución Poisson con $\langle k\rangle = 12$

En una distribución Poisson la varianza es igual a la media: $\sigma_k^2 = \langle k\rangle$. Por lo tanto

$$
\langle k^2\rangle = \sigma_k^2 + \langle k\rangle^2 = \langle k\rangle + \langle k\rangle^2 = 12 + 144 = 156
$$

**Umbral con la fórmula de red heterogénea:**

$$
\lambda_c = \frac{\langle k\rangle}{\langle k^2\rangle - \langle k\rangle} = \frac{12}{156 - 12} = \frac{12}{144} = \frac{1}{12} = 0.0833
$$

$$
\frac{\beta}{\gamma} = 1.6 > 0.0833 \;\Longrightarrow\; \text{hay epidemia}\quad \left(R_0^{\text{red}} = 1.6 \times 12 = 19.2\right)
$$

**SIR clásico:**

$$
R_0 = \frac{\beta\langle k\rangle}{\gamma} = \frac{0.08 \times 12}{0.05} = 19.2 > 1
\quad\Longleftrightarrow\quad
\frac{\beta}{\gamma} > \frac{1}{\langle k\rangle} = 0.0833
$$

**¿Son iguales? Sí, y no es casualidad.** Para una distribución Poisson, $\langle k^2\rangle - \langle k\rangle = \langle k\rangle^2$, así que la fórmula de red se reduce exactamente a la condición clásica:

$$
\lambda_c = \frac{\langle k\rangle}{\langle k\rangle^2} = \frac{1}{\langle k\rangle}
\qquad\text{y}\qquad
R_0^{\text{red}} = \frac{\beta}{\gamma}\cdot\frac{\langle k\rangle^2}{\langle k\rangle} = \frac{\beta\langle k\rangle}{\gamma} = R_0^{\text{clásico}}
$$

La red Erdős-Rényi con distribución Poisson es la versión en red de la **mezcla homogénea**. Todos los nodos tienen un grado parecido al promedio (coeficiente de variación $1/\sqrt{12} = 0.29$) y no hay hubs, así que el supuesto del SIR clásico de que "todos tienen $\langle k\rangle$ contactos" es una buena aproximación.

En general, la fórmula de red **generaliza** a la clásica. Las dos coinciden si y solo si $\sigma_k^2 = \langle k\rangle$. Si $\sigma_k^2 > \langle k\rangle$ (red heterogénea), $\lambda_c < 1/\langle k\rangle$ y el SIR clásico **subestima** el riesgo de brote, que es justo lo que pasó con el SARS en 2003.

In [15]:
kX = 12
k2X = kX + kX**2
print(f"<k²>_X = <k> + <k>² = {k2X}")
lam_X = reporte("Municipio X (red, Poisson)", beta, gamma_rec, kX, k2X)
R0_clasico = beta * kX / gamma_rec
print(f"\nSIR clásico: R0 = β<k>/γ = {R0_clasico:.2f}  ⇔  β/γ > 1/<k> = {1/kX:.5f}")
print(f"Red:         R0_red = (β/γ)(<k²>-<k>)/<k> = {(beta/gamma_rec)*(k2X-kX)/kX:.2f}")

<k²>_X = <k> + <k>² = 156
Municipio X (red, Poisson)         <k>= 12.0000  <k²>= 156.0000  λc=0.08333  β/γ=1.6000  (β/γ)/λc=  19.20  → SE PROPAGA

SIR clásico: R0 = β<k>/γ = 19.20  ⇔  β/γ > 1/<k> = 0.08333
Red:         R0_red = (β/γ)(<k²>-<k>)/<k> = 19.20


### b. Municipio Y: $P(k) \sim k^{-2.4}$, $k_{\min} = 1$, $k_{\max} = 800$

In [16]:
gY, kmin, kmax = 2.4, 1, 800

# Fórmula tal como aparece en el enunciado (signo incorrecto)
k2_literal = (gY - 1) / (gY - 3) * kmin**2 * (kmax / kmin)**(3 - gY)
# Fórmula con el signo corregido
k2Y = (gY - 1) / (3 - gY) * kmin**2 * (kmax / kmin)**(3 - gY)
k1Y = (gY - 1) / (gY - 2) * kmin

print(f"(γ-1)/(γ-3)     = {(gY-1)/(gY-3):.4f}   -> <k²> literal = {k2_literal:.2f}  (imposible, < 0)")
print(f"(kmax/kmin)^0.6 = {(kmax/kmin)**(3-gY):.4f}")
print(f"<k²> corregido  = {k2Y:.4f}")
print(f"<k>  continuo   = {k1Y:.4f}\n")
lam_Y = reporte("Municipio Y (aprox. continua)", beta, gamma_rec, k1Y, k2Y)

(γ-1)/(γ-3)     = -2.3333   -> <k²> literal = -128.77  (imposible, < 0)
(kmax/kmin)^0.6 = 55.1892
<k²> corregido  = 128.7748
<k>  continuo   = 3.5000

Municipio Y (aprox. continua)      <k>=  3.5000  <k²>= 128.7748  λc=0.02794  β/γ=1.6000  (β/γ)/λc=  57.27  → SE PROPAGA


In [17]:
def momentos_continuos_ley_potencias(g, kmin, kmax):
    # <k> y <k²> exactos para P(k) = C k^-g en [kmin, kmax] (continuo, con corte finito)
    C = (g - 1) / (kmin**(1 - g) - kmax**(1 - g))
    m1 = C * (kmin**(2 - g) - kmax**(2 - g)) / (g - 2)
    m2 = C * (kmax**(3 - g) - kmin**(3 - g)) / (3 - g)
    return m1, m2


def distribucion_ley_potencias(g, kmin, kmax):
    # P(k) ∝ k^-g discreta para k = kmin..kmax
    k = np.arange(kmin, kmax + 1, dtype=float)
    p = k**(-g)
    return k, p / p.sum()


def momentos(k, p):
    return float((k * p).sum()), float((k**2 * p).sum())


kY, pY = distribucion_ley_potencias(gY, kmin, kmax)
m1c, m2c = momentos_continuos_ley_potencias(gY, kmin, kmax)
m1d, m2d = momentos(kY, pY)

reporte("Y: fórmula del curso (corregida)", beta, gamma_rec, k1Y, k2Y)
reporte("Y: continua con corte k_max", beta, gamma_rec, m1c, m2c)
reporte("Y: discreta exacta k=1..800", beta, gamma_rec, m1d, m2d)
print(f"\nEn la versión discreta, P(k=1) = {pY[0]:.3f}: la mayoría de nodos tiene un solo contacto.")

Y: fórmula del curso (corregida)   <k>=  3.5000  <k²>= 128.7748  λc=0.02794  β/γ=1.6000  (β/γ)/λc=  57.27  → SE PROPAGA
Y: continua con corte k_max        <k>=  3.2588  <k²>= 126.4523  λc=0.02645  β/γ=1.6000  (β/γ)/λc=  60.48  → SE PROPAGA
Y: discreta exacta k=1..800        <k>=  2.1204  <k²>=  65.7001  λc=0.03335  β/γ=1.6000  (β/γ)/λc=  47.98  → SE PROPAGA

En la versión discreta, P(k=1) = 0.723: la mayoría de nodos tiene un solo contacto.


In [18]:
print(f"{'k_max':>10}{'<k>':>9}{'<k²>':>12}{'λc':>10}")
for km in [50, 100, 800, 10_000, 1_000_000]:
    m1, m2 = momentos_continuos_ley_potencias(gY, kmin, km)
    print(f"{km:>10}{m1:>9.3f}{m2:>12.2f}{umbral_epidemico(m1, m2):>10.5f}")

     k_max      <k>        <k²>        λc
        50    2.780       22.16   0.14345
       100    2.950       34.70   0.09290
       800    3.259      126.45   0.02645
     10000    3.412      583.77   0.00588
   1000000    3.486     9286.83   0.00038


### c. Vacunación del 20 %: aleatoria contra dirigida a los hubs

In [19]:
def distribucion_poisson(media, kmax=100):
    k = np.arange(kmax + 1, dtype=float)
    logp = -media + k * math.log(media) - np.array([math.lgamma(x + 1) for x in k])
    p = np.exp(logp)
    return k, p / p.sum()


def diluir(k, p, q):
    # Momentos cuando cada arista sobrevive con probabilidad q (dilución binomial del grado)
    m1, m2 = momentos(k, p)
    return q * m1, q**2 * m2 + q * (1 - q) * m1


def vacunacion_aleatoria(k, p, g):
    return diluir(k, p, 1 - g), g


def vacunacion_dirigida(k, p, g):
    # Vacuna la fracción g de nodos de mayor grado; retorna momentos efectivos y f_e
    vacunados = np.zeros_like(p)
    restante = g
    for i in range(len(k) - 1, -1, -1):  # de mayor a menor grado
        tomar = min(p[i], restante)
        vacunados[i] = tomar
        restante -= tomar
        if restante <= 1e-15:
            break
    p_rest = p - vacunados
    f_e = (k * vacunados).sum() / (k * p).sum()
    return diluir(k, p_rest / p_rest.sum(), 1 - f_e), f_e


def g_critico(estrategia, k, p, razon, tol=1e-6):
    # Menor fracción g tal que λc(g) > β/γ (bisección; λc crece con g)
    lo, hi = 0.0, 0.999999
    if umbral_epidemico(*estrategia(k, p, hi)[0]) <= razon:
        return float("nan")
    while hi - lo > tol:
        mid = (lo + hi) / 2
        if umbral_epidemico(*estrategia(k, p, mid)[0]) > razon:
            hi = mid
        else:
            lo = mid
    return hi

In [20]:
kX_dist, pX = distribucion_poisson(12)
razon = beta / gamma_rec
g_vac = 0.20

filas = []
for municipio, (k_, p_) in {"X (Poisson)": (kX_dist, pX), "Y (k^-2.4)": (kY, pY)}.items():
    m_base = momentos(k_, p_)
    (m_ale, q_ale) = vacunacion_aleatoria(k_, p_, g_vac)
    (m_dir, f_e) = vacunacion_dirigida(k_, p_, g_vac)
    for estrategia, m, aristas_perdidas in [("Sin vacunar", m_base, 0.0),
                                            ("Aleatoria 20 %", m_ale, q_ale),
                                            ("Dirigida 20 %", m_dir, f_e)]:
        lam = umbral_epidemico(*m)
        filas.append((municipio, estrategia, *m, aristas_perdidas, lam, razon / lam,
                      m[0] / m_base[0] - 1, m[1] / m_base[1] - 1))

print(f"{'Municipio':<13}{'Estrategia':<16}{'<k>':>8}{'<k²>':>9}{'Δ<k>':>8}{'Δ<k²>':>8}"
      f"{'aristas':>9}{'λc':>9}{'(β/γ)/λc':>10}  Resultado")
print(f"{'':<29}{'':>8}{'':>9}{'':>8}{'':>8}{'perdidas':>9}")
for mun, est, m1, m2, fe, lam, r, d1, d2 in filas:
    res = "se propaga" if r > 1 else "CONTENIDO"
    print(f"{mun:<13}{est:<16}{m1:>8.3f}{m2:>9.3f}{d1:>8.1%}{d2:>8.1%}{fe:>9.1%}{lam:>9.4f}{r:>10.2f}  {res}")

Municipio    Estrategia           <k>     <k²>    Δ<k>   Δ<k²>  aristas       λc  (β/γ)/λc  Resultado
                                                               perdidas
X (Poisson)  Sin vacunar       12.000  156.000    0.0%    0.0%     0.0%   0.0833     19.20  se propaga
X (Poisson)  Aleatoria 20 %     9.600  101.760  -20.0%  -34.8%    20.0%   0.1042     15.36  se propaga
X (Poisson)  Dirigida 20 %      7.701   64.695  -35.8%  -58.5%    28.3%   0.1351     11.84  se propaga
Y (k^-2.4)   Sin vacunar        2.120   65.700    0.0%    0.0%     0.0%   0.0334     47.98  se propaga
Y (k^-2.4)   Aleatoria 20 %     1.696   42.387  -20.0%  -35.5%    20.0%   0.0417     38.38  se propaga
Y (k^-2.4)   Dirigida 20 %      0.453    0.486  -78.6%  -99.3%    58.6%  13.7544      0.12  CONTENIDO


In [21]:
print("Fracción mínima a vacunar para contener el brote (β/γ = 1.6):\n")
print(f"{'Municipio':<13}{'Aleatoria':>12}{'Dirigida':>12}")
for municipio, (k_, p_) in {"X (Poisson)": (kX_dist, pX), "Y (k^-2.4)": (kY, pY)}.items():
    ga = g_critico(vacunacion_aleatoria, k_, p_, razon)
    gd = g_critico(vacunacion_dirigida, k_, p_, razon)
    print(f"{municipio:<13}{ga:>12.1%}{gd:>12.1%}")

# Referencia analítica (continua) para Y: fracción de aristas eliminadas al vacunar el 20 % con mayor grado
print(f"\nY continuo: grado de corte k_c = g^(-1/(γ-1)) = {g_vac**(-1/(gY-1)):.2f}; "
      f"aristas eliminadas f_e = g^((γ-2)/(γ-1)) = {g_vac**((gY-2)/(gY-1)):.1%}")

Fracción mínima a vacunar para contener el brote (β/γ = 1.6):

Municipio       Aleatoria    Dirigida
X (Poisson)         94.8%       83.2%
Y (k^-2.4)          97.9%        4.3%

Y continuo: grado de corte k_c = g^(-1/(γ-1)) = 3.16; aristas eliminadas f_e = g^((γ-2)/(γ-1)) = 63.1%
